In [1]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import lr_scheduler
import torch.nn as nn


seed = 1234
np.random.seed(seed)
pd.set_option('display.max_columns', None)

In [2]:
!pip install --upgrade sympy

# Deep Neural Network (PyTorch)
## Author: Kaveh Zare

In [3]:
"""
Scaling & Splitting Data
"""
df_tr = pd.read_csv('clean_training_data.csv')
df_te = pd.read_csv('clean_testing_data.csv')

X_tr = df_tr.drop(columns=["readmitted"])
y_tr = df_tr["readmitted"]

X_te = df_te.drop(columns=["readmitted"])
y_te = df_te["readmitted"]

X_train_raw, X_val_raw, y_tr, y_val = train_test_split(
    X_tr, y_tr, train_size=0.9, test_size=0.1, shuffle=True, random_state=seed
)

In [4]:
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_train_raw)

X_val_scaled = scaler.transform(X_val_raw)
X_te_scaled = scaler.transform(X_te) # X_te comes from your second csv

In [5]:
X_train_tensor = torch.tensor(X_tr_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_tr.values, dtype=torch.long)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.long)

X_test_tensor = torch.tensor(X_te_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_te.values, dtype=torch.long)

In [8]:
class DeepClassifier(nn.Module):
    def __init__(self, input_size, num_classes):
        super(DeepClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)

# 3. Initialization
model = DeepClassifier(input_size=X_train_tensor.shape[1], num_classes=3)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)

In [9]:
epochs = 50
best_val_loss = float('inf')
best_val_acc = 0.0
patience_counter = 0
early_stopping_patience = 7
min_epochs = 5

for epoch in range(epochs):
    model.train()
    train_loss, train_corrects, train_total = 0.0, 0, 0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_X.size(0)
        _, predicted = torch.max(outputs, 1)
        train_corrects += (predicted == batch_y).sum().item()
        train_total += batch_y.size(0)

    model.eval()
    val_loss, val_corrects, val_total = 0.0, 0, 0

    with torch.no_grad():
        val_outputs = model(X_test_tensor)
        v_loss = criterion(val_outputs, y_test_tensor)

        val_loss = v_loss.item()
        _, val_predicted = torch.max(val_outputs, 1)
        val_corrects = (val_predicted == y_test_tensor).sum().item()
        val_total = y_test_tensor.size(0)

    train_acc = (train_corrects / train_total) * 100
    val_acc = (val_corrects / val_total) * 100
    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1:03d} | LR: {current_lr:.6f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        patience_counter = 0
        print(f"  --> Best model saved (Val Loss: {val_loss:.4f})")
    else:
        if epoch >= min_epochs:
            patience_counter += 1

    if patience_counter >= early_stopping_patience:
        print(f"Stopping early. No improvement for {early_stopping_patience} epochs.")
        break

model.load_state_dict(torch.load('best_model.pth'))
print(f"\nFinal Best Validation Accuracy: {best_val_acc:.2f}%")

Epoch 001 | LR: 0.001000 | Train Acc: 57.40% | Val Acc: 58.25% | Val Loss: 0.8733
  --> Best model saved (Val Loss: 0.8733)
Epoch 002 | LR: 0.001000 | Train Acc: 58.65% | Val Acc: 58.78% | Val Loss: 0.8730
  --> Best model saved (Val Loss: 0.8730)
Epoch 003 | LR: 0.001000 | Train Acc: 58.72% | Val Acc: 58.93% | Val Loss: 0.8737
Epoch 004 | LR: 0.001000 | Train Acc: 58.84% | Val Acc: 58.47% | Val Loss: 0.8711
  --> Best model saved (Val Loss: 0.8711)
Epoch 005 | LR: 0.001000 | Train Acc: 59.08% | Val Acc: 58.68% | Val Loss: 0.8687
  --> Best model saved (Val Loss: 0.8687)
Epoch 006 | LR: 0.001000 | Train Acc: 59.16% | Val Acc: 59.07% | Val Loss: 0.8680
  --> Best model saved (Val Loss: 0.8680)
Epoch 007 | LR: 0.001000 | Train Acc: 59.06% | Val Acc: 58.59% | Val Loss: 0.8682
Epoch 008 | LR: 0.001000 | Train Acc: 59.06% | Val Acc: 59.20% | Val Loss: 0.8653
  --> Best model saved (Val Loss: 0.8653)
Epoch 009 | LR: 0.001000 | Train Acc: 59.23% | Val Acc: 58.76% | Val Loss: 0.8703
Epoch 010 

In [10]:

model.eval()

with torch.no_grad():

    test_outputs = model(X_test_tensor)

    test_loss = criterion(test_outputs, y_test_tensor).item()

    _, test_predicted = torch.max(test_outputs, 1)
    test_correct = (test_predicted == y_test_tensor).sum().item()
    test_total = y_test_tensor.size(0)
    test_acc = (test_correct / test_total) * 100

print("-" * 30)
print(f"Final Test Loss:     {test_loss:.4f}")
print(f"Final Test Accuracy: {test_acc:.2f}%")
print("-" * 30)

------------------------------
Final Test Loss:     0.8607
Final Test Accuracy: 59.30%
------------------------------
